In [1]:
import requests
import pandas as pd
import json

# Testando conexão com a API do IBGE
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
resposta = requests.get(url)

print(f"Status: {resposta.status_code}")
print(f"Tipo de dado: {type(resposta.json())}")
print(f"Quantidade de estados: {len(resposta.json())}")
print(f"\nPrimeiro estado: {resposta.json()[0]}")


Status: 200
Tipo de dado: <class 'list'>
Quantidade de estados: 27

Primeiro estado: {'id': 11, 'sigla': 'RO', 'nome': 'Rondônia', 'regiao': {'id': 1, 'sigla': 'N', 'nome': 'Norte'}}


In [2]:
# Buscando todos os estados
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
resposta = requests.get(url)
estados = resposta.json()

# Transformando em DataFrame
df_estados = pd.DataFrame(estados)
print(df_estados.head())
print(f"\nShape: {df_estados.shape}")
print(f"\nColunas: {df_estados.columns.tolist()}")

   id sigla      nome                                    regiao
0  11    RO  Rondônia  {'id': 1, 'sigla': 'N', 'nome': 'Norte'}
1  12    AC      Acre  {'id': 1, 'sigla': 'N', 'nome': 'Norte'}
2  13    AM  Amazonas  {'id': 1, 'sigla': 'N', 'nome': 'Norte'}
3  14    RR   Roraima  {'id': 1, 'sigla': 'N', 'nome': 'Norte'}
4  15    PA      Pará  {'id': 1, 'sigla': 'N', 'nome': 'Norte'}

Shape: (27, 4)

Colunas: ['id', 'sigla', 'nome', 'regiao']


In [3]:
# A coluna 'regiao' é um dicionário dentro do DataFrame
# Precisamos extrair os dados dela

df_estados["regiao_nome"] = df_estados["regiao"].apply(lambda x: x["nome"])
df_estados["regiao_sigla"] = df_estados["regiao"].apply(lambda x: x["sigla"])

# Selecionando colunas relevantes
df_limpo = df_estados[["id", "sigla", "nome", "regiao_nome", "regiao_sigla"]]
df_limpo = df_limpo.sort_values("nome").reset_index(drop=True)

print(df_limpo)
print(f"\nRegiões únicas: {df_limpo['regiao_nome'].unique()}")

    id sigla                 nome   regiao_nome regiao_sigla
0   12    AC                 Acre         Norte            N
1   27    AL              Alagoas      Nordeste           NE
2   16    AP                Amapá         Norte            N
3   13    AM             Amazonas         Norte            N
4   29    BA                Bahia      Nordeste           NE
5   23    CE                Ceará      Nordeste           NE
6   53    DF     Distrito Federal  Centro-Oeste           CO
7   32    ES       Espírito Santo       Sudeste           SE
8   52    GO                Goiás  Centro-Oeste           CO
9   21    MA             Maranhão      Nordeste           NE
10  51    MT          Mato Grosso  Centro-Oeste           CO
11  50    MS   Mato Grosso do Sul  Centro-Oeste           CO
12  31    MG         Minas Gerais       Sudeste           SE
13  41    PR               Paraná           Sul            S
14  25    PB              Paraíba      Nordeste           NE
15  15    PA            

In [4]:
# Quantos estados por região?
print("Estados por região:")
print(df_limpo["regiao_nome"].value_counts())

# Filtrando só o Nordeste
nordeste = df_limpo[df_limpo["regiao_nome"] == "Nordeste"]
print(f"\nEstados do Nordeste:")
print(nordeste[["sigla", "nome"]].to_string(index=False))

# Rio de Janeiro está em qual região?
rj = df_limpo[df_limpo["sigla"] == "RJ"]
print(f"\nRio de Janeiro:")
print(rj[["nome", "regiao_nome"]].to_string(index=False))

Estados por região:
regiao_nome
Nordeste        9
Norte           7
Centro-Oeste    4
Sudeste         4
Sul             3
Name: count, dtype: int64

Estados do Nordeste:
sigla                nome
   AL             Alagoas
   BA               Bahia
   CE               Ceará
   MA            Maranhão
   PB             Paraíba
   PE          Pernambuco
   PI               Piauí
   RN Rio Grande do Norte
   SE             Sergipe

Rio de Janeiro:
          nome regiao_nome
Rio de Janeiro     Sudeste


In [5]:
# Cotação do dólar — API oficial do Banco Central
import datetime

hoje = datetime.date.today().strftime("%m-%d-%Y")
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarDia(dataCotacao=@dataCotacao)?@dataCotacao='{hoje}'&$format=json"

resposta = requests.get(url)
dados = resposta.json()

if dados["value"]:
    cotacao = dados["value"][0]
    print(f"Dólar hoje ({hoje}):")
    print(f"Compra:  R${cotacao['cotacaoCompra']:.4f}")
    print(f"Venda:   R${cotacao['cotacaoVenda']:.4f}")
else:
    print("Mercado fechado hoje — sem cotação disponível")
    print("(fins de semana e feriados não têm cotação)")

Dólar hoje (07-29-2026):
Compra:  R$5.1211
Venda:   R$5.1217


In [6]:
# Salvando o DataFrame em CSV na pasta data/processed
df_limpo.to_csv("data/processed/estados_brasileiros.csv", index=False)
print("Arquivo salvo!")

# Verificando
df_verificacao = pd.read_csv("data/processed/estados_brasileiros.csv")
print(f"Linhas: {df_verificacao.shape[0]}")
print(df_verificacao.head())

OSError: Cannot save file into a non-existent directory: 'data\processed'

In [7]:
import os

# Criando a pasta se não existir
os.makedirs("data/processed", exist_ok=True)

# Agora salva
df_limpo.to_csv("data/processed/estados_brasileiros.csv", index=False)
print("Arquivo salvo!")

# Verificando
df_verificacao = pd.read_csv("data/processed/estados_brasileiros.csv")
print(f"Linhas: {df_verificacao.shape[0]}")
print(df_verificacao.head())

Arquivo salvo!
Linhas: 27
   id sigla      nome regiao_nome regiao_sigla
0  12    AC      Acre       Norte            N
1  27    AL   Alagoas    Nordeste           NE
2  16    AP     Amapá       Norte            N
3  13    AM  Amazonas       Norte            N
4  29    BA     Bahia    Nordeste           NE


In [8]:
# Salvando o DataFrame em CSV na pasta data/processed
df_limpo.to_csv("data/processed/estados_brasileiros.csv", index=False)
print("Arquivo salvo!")

# Verificando
df_verificacao = pd.read_csv("data/processed/estados_brasileiros.csv")
print(f"Linhas: {df_verificacao.shape[0]}")
print(df_verificacao.head())

Arquivo salvo!
Linhas: 27
   id sigla      nome regiao_nome regiao_sigla
0  12    AC      Acre       Norte            N
1  27    AL   Alagoas    Nordeste           NE
2  16    AP     Amapá       Norte            N
3  13    AM  Amazonas       Norte            N
4  29    BA     Bahia    Nordeste           NE
